In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# 假设序列有 4 个"词"，每个词用 8 维向量表示
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)   # 4个问题
K = torch.randn(seq_len, d_k)   # 4个标签
V = torch.randn(seq_len, d_k)   # 4份内容

print("Q:", Q.shape, " K:", K.shape, " V:", V.shape)

Q: torch.Size([4, 8])  K: torch.Size([4, 8])  V: torch.Size([4, 8])


In [2]:
# 第1步：相似度 —— 每个Query和每个Key点积
scores = Q @ K.T                    # [4,4] @ [4,4] → [4,4]
print("相似度矩阵 scores:\n", scores)

# 第2步：缩放
scores = scores / (d_k ** 0.5)

# 第3步：softmax，让每一行加起来=1
weights = F.softmax(scores, dim=-1)  # dim=-1: 对每一行做归一化
print("注意力权重 weights:\n", weights)
print("每行之和:", weights.sum(dim=-1))   # 应全是 1.0

# 第4步：加权求和
out = weights @ V                    # [4,4] @ [4,8] → [4,8]
print("输出 out:", out.shape)        # [4,8]，与 V 同形

相似度矩阵 scores:
 tensor([[-0.0146,  5.1091, -0.3921, -3.7779],
        [ 0.4648,  0.5446, -0.7034,  0.9571],
        [ 1.3816, -7.0338, -0.2771,  2.2690],
        [ 0.6756,  3.8463, -1.2603, -2.3062]])
注意力权重 weights:
 tensor([[0.1211, 0.7410, 0.1060, 0.0320],
        [0.2577, 0.2651, 0.1705, 0.3067],
        [0.3360, 0.0171, 0.1869, 0.4599],
        [0.2032, 0.6235, 0.1025, 0.0708]])
每行之和: tensor([1.0000, 1.0000, 1.0000, 1.0000])
输出 out: torch.Size([4, 8])


In [3]:
import torch.nn as nn

x = torch.randn(4, 8)               # 4个词，各8维（这一步现实中是词嵌入）

# 三个线性层，把同一个 x 投影成 Q、K、V
W_q = nn.Linear(8, 8, bias=False)
W_k = nn.Linear(8, 8, bias=False)
W_v = nn.Linear(8, 8, bias=False)

Q, K, V = W_q(x), W_k(x), W_v(x)

# 复用上面的四步计算
scores  = Q @ K.T / (8 ** 0.5)
weights = F.softmax(scores, dim=-1)
out     = weights @ V
print("自注意力输出:", out.shape)   # [4,8]
print("注意力权重:\n", weights.round(decimals=2))

自注意力输出: torch.Size([4, 8])
注意力权重:
 tensor([[0.2900, 0.2800, 0.3300, 0.1100],
        [0.2500, 0.2200, 0.2500, 0.2800],
        [0.3000, 0.2600, 0.1700, 0.2800],
        [0.2300, 0.3900, 0.1900, 0.1900]], grad_fn=<RoundBackward1>)
